<a href="https://colab.research.google.com/github/rj-Fariha/rag-knowledge-assistant-eval/blob/main/rag_knowledge_assistant_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# RAG Knowledge Assistant with Evaluation
# Retrieval (sentence embeddings) + Generation (small local model)
# 100% free, no API keys, runs on CPU. Models download once (~300MB
# total) then run locally - no rate limits, no internet needed after.
# ============================================================

!pip install sentence-transformers transformers -q
print("Installed")

Installed


In [ ]:
# ------------------------------------------------------------
# STEP 1: Build a small knowledge base (company policy documents)
# and a labeled set of test questions with known correct answers
# and known correct source paragraph - this is what lets us
# MEASURE retrieval and answer quality later.
# ------------------------------------------------------------

documents = [
    "Employees are entitled to 21 days of paid annual leave per calendar year. Leave requests must be submitted at least 5 working days in advance through the HR portal.",
    "Remote work is permitted up to 3 days per week, subject to manager approval. Employees must be reachable during core hours of 10 AM to 4 PM.",
    "Expense reimbursements must be submitted within 30 days of the expense being incurred, with an original receipt attached. Reimbursements are processed within 10 business days.",
    "New employees undergo a 90-day probation period, during which performance is reviewed at the 30, 60, and 90 day marks by their direct manager.",
    "The company provides health insurance coverage for employees and their immediate family, effective from the first day of employment.",
    "Sick leave is separate from annual leave, with employees entitled to 14 days of paid sick leave per year. A medical certificate is required for absences longer than 2 consecutive days.",
    "Employees are eligible for a performance bonus annually, calculated based on individual performance rating and overall company performance for the fiscal year.",
    "The standard notice period for resignation is 30 days for employees who have completed probation, and 7 days for those still within the probation period.",
    "Maternity leave is provided for 90 days at full pay, and paternity leave is provided for 10 days at full pay, both effective from the confirmed date of employment.",
    "Employees are provided a laptop and necessary equipment for remote work, which must be returned upon termination of employment.",
]

# Test questions: each paired with the paragraph index that contains
# the answer (for measuring retrieval), and a keyword expected in a
# correct answer (for measuring answer quality).
test_questions = [
    {"question": "How many days of annual leave do employees get?", "correct_doc_idx": 0, "expected_keyword": "21"},
    {"question": "How many days can I work remotely per week?", "correct_doc_idx": 1, "expected_keyword": "3"},
    {"question": "How long do I have to submit an expense reimbursement?", "correct_doc_idx": 2, "expected_keyword": "30"},
    {"question": "How long is the probation period for new employees?", "correct_doc_idx": 3, "expected_keyword": "90"},
    {"question": "When does health insurance coverage start?", "correct_doc_idx": 4, "expected_keyword": "first day"},
    {"question": "How many sick leave days am I entitled to?", "correct_doc_idx": 5, "expected_keyword": "14"},
    {"question": "What is the resignation notice period after probation?", "correct_doc_idx": 7, "expected_keyword": "30"},
    {"question": "How many days of maternity leave are provided?", "correct_doc_idx": 8, "expected_keyword": "90"},
]

print(f"Knowledge base: {len(documents)} documents")
print(f"Test set: {len(test_questions)} labeled questions")

Knowledge base: 10 documents
Test set: 8 labeled questions


In [ ]:
# ------------------------------------------------------------
# STEP 2: Turn every document into a vector (embedding), so that
# meaning becomes something we can measure distance between.
# Sentences with similar meaning end up as similar vectors, even
# if they don't share exact words.
# ------------------------------------------------------------

from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2")  # small, fast, free, downloads once (~80MB)

doc_embeddings = embedder.encode(documents)
print(f"Embedded {len(documents)} documents into vectors of size {doc_embeddings.shape[1]}")

def retrieve(question: str, top_k: int = 2):
    """Find the top_k documents most similar in meaning to the question."""
    q_embedding = embedder.encode([question])[0]
    # cosine similarity = how aligned two vectors are, ignoring their length
    similarities = np.dot(doc_embeddings, q_embedding) / (
        np.linalg.norm(doc_embeddings, axis=1) * np.linalg.norm(q_embedding)
    )
    top_indices = np.argsort(similarities)[::-1][:top_k]
    return [(idx, documents[idx], similarities[idx]) for idx in top_indices]

# Test it
sample_q = test_questions[0]["question"]
print(f"\nQuestion: {sample_q}")
for idx, doc, score in retrieve(sample_q):
    print(f"  [{score:.3f}] doc #{idx}: {doc[:80]}...")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedded 10 documents into vectors of size 384

Question: How many days of annual leave do employees get?
  [0.817] doc #0: Employees are entitled to 21 days of paid annual leave per calendar year. Leave ...
  [0.630] doc #5: Sick leave is separate from annual leave, with employees entitled to 14 days of ...


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

def answer_question(question: str, top_k: int = 2):
    retrieved = retrieve(question, top_k=top_k)
    context = " ".join([doc for _, doc, _ in retrieved])

    prompt = f"Answer the question using only the context below.\n\nContext: {context}\n\nQuestion: {question}\n\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    outputs = model.generate(**inputs, max_new_tokens=50)
    answer_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return {
        "question": question,
        "answer": answer_text,
        "retrieved_doc_indices": [idx for idx, _, _ in retrieved],
        "context_used": context,
    }

# Test it end to end
result = answer_question(test_questions[0]["question"])
print("Question:", result["question"])
print("Answer:  ", result["answer"])
print("Retrieved doc indices:", result["retrieved_doc_indices"])

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Question: How many days of annual leave do employees get?
Answer:   21
Retrieved doc indices: [np.int64(0), np.int64(5)]


In [ ]:
# ------------------------------------------------------------
# STEP 4: Evaluate the whole pipeline on the labeled test set.
# Two separate metrics, because retrieval and generation can each
# fail independently - measuring them separately tells you WHICH
# part of the system needs work if accuracy is low.
# ------------------------------------------------------------

retrieval_correct = 0
answer_correct = 0
results_log = []

for item in test_questions:
    result = answer_question(item["question"], top_k=2)

    retrieved_ok = item["correct_doc_idx"] in result["retrieved_doc_indices"]
    answer_ok = item["expected_keyword"].lower() in result["answer"].lower()

    retrieval_correct += retrieved_ok
    answer_correct += answer_ok

    results_log.append({
        "question": item["question"],
        "answer": result["answer"],
        "retrieval_correct": retrieved_ok,
        "answer_correct": answer_ok,
    })

n = len(test_questions)
print(f"Retrieval precision@2: {retrieval_correct}/{n} ({retrieval_correct/n:.1%})")
print(f"Answer correctness:    {answer_correct}/{n} ({answer_correct/n:.1%})")

print("\n--- Per-question results ---")
for r in results_log:
    status = "OK" if r["answer_correct"] else "MISS"
    print(f"[{status}] Q: {r['question']}\n     A: {r['answer']}")

Retrieval precision@2: 8/8 (100.0%)
Answer correctness:    8/8 (100.0%)

--- Per-question results ---
[OK] Q: How many days of annual leave do employees get?
     A: 21
[OK] Q: How many days can I work remotely per week?
     A: 3
[OK] Q: How long do I have to submit an expense reimbursement?
     A: 30 days
[OK] Q: How long is the probation period for new employees?
     A: 90-day
[OK] Q: When does health insurance coverage start?
     A: first day of employment
[OK] Q: How many sick leave days am I entitled to?
     A: 14 days
[OK] Q: What is the resignation notice period after probation?
     A: 30 days
[OK] Q: How many days of maternity leave are provided?
     A: 90
